# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("Week3_Rohan")  # from Colab Secrets

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Register the HF token properly as a DuckDB secret
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("HF secret registered.")
print(bool(HF_TOKEN), len(HF_TOKEN) if HF_TOKEN else 0)

HF secret registered.
True 37


**Unit of analysis:** one row = one content page, on one specific day
(`report_date × client_hash_id × content_hash_id`), from `fact_content_daily_performance`.

**Time window:** I'm developing on a mid-panel month, `month=2026-03`, per the warehouse skill's
warning — the `_sample` table is the FINAL month (June 2026) only, which would be the natural
outcome window of any past→future label. Using it now would mean developing my label inside my
own test window, so I'm avoiding it entirely for this exercise.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature candidates:**
- `gsc_impressions` — search visibility, observed daily
- `gsc_clicks` — observed click count
- `gsc_avg_position` — observed search ranking position
- `ga4_sessions` — observed analytics traffic
- `ga4_engagement_rate` — observed engagement behavior

**Label / proxy:**
A future decline flag I would define myself from a forward time window (not built yet — this
notebook is the contract step, before modeling begins).

**Context (not features, not labels):**
- `client_hash_id`, `content_hash_id` — pseudonymous join/grouping keys only, never features
- `report_date` — defines the time window, not itself a predictive input

**Excluded, deliberately:**
Any pre-existing product decision flag (`health_score`, `priority_score`, `action_type`). These
aren't shipped in this dataset on purpose — using them (or a rebuilt version of them) as a feature
would mean the model copies an existing rule instead of discovering a real signal from evidence.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q_grain = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
con.sql(q_grain).show()  # empty result = grain confirmed

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



Query 1 proves the grain: one row really is one content page, on one day, for one client.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Named limitation:** client history in this warehouse is an unbalanced panel. Different
clients began tracking at very different times, and only 9 of 70 clients have 12+ months of
history. This means any trend or seasonality analysis is only reliable for a subset of clients —
results from `month=2026-03` won't necessarily generalize evenly across all clients, since some
have thin or missing history for that period. I've filtered on `ga4_data_available IS TRUE` to
avoid mistaking "not tracked yet" for "zero engagement," but the underlying imbalance in history
depth remains a real constraint on how confidently I can generalize any pattern found here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.